# LeetCode 572 — Subtree of Another Tree bằng Python

**Link đề:** https://leetcode.com/problems/subtree-of-another-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, String Matching, Binary Tree, Hash Function

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def isSubtree(self, root: TreeNode | None, subRoot: TreeNode | None) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu chính xác định nghĩa **cây con** (subtree): một node **cùng toàn bộ hậu duệ** của nó — không được cắt bớt.
2. Viết hàm `is_same(a, b)` so sánh hai cây (LeetCode 100) và ghép nó vào bài này.
3. Đi từ lời giải đệ quy `O(m·n)` lên ý tưởng **tuần tự hóa cây thành chuỗi** (string matching) và **băm cây** (Merkle hashing) gần `O(m + n)`.
4. Nhận ra các bẫy: thiếu dấu `null`, `"2"` khớp nhầm trong `"12"`, cây con "giống phần đầu" nhưng có thêm con.
5. Trace, chứng minh đúng, test đối chiếu trên cây ngẫu nhiên.

## 1. Đề bài

Cho hai cây nhị phân `root` và `subRoot`. Trả về `True` nếu trong `root` có một **cây con** có **cùng cấu trúc và cùng giá trị** với `subRoot`; ngược lại `False`.

**Cây con** của cây `T` = một node trong `T` **cùng tất cả hậu duệ** của node đó. Bản thân `T` cũng là cây con của chính nó.

```text
Ví dụ 1:   root          subRoot          Ví dụ 2:   root            subRoot
              3              4                         3                4
             / \            / \                       / \              / \
            4   5          1   2                     4   5            1   2
           / \                                      / \
          1   2                                    1   2
                                                      /
                                                     0
           -> True                                   -> False
```

Ví dụ 2 là **bẫy**: phần trên của cây con gốc `4` giống `subRoot`, nhưng node `2` có thêm con `0` → cây con gốc `4` là `[4,1,2,null,null,0]` ≠ `[4,1,2]`.

| `root` | `subRoot` | Output |
|---|---|:---:|
| `[3,4,5,1,2]` | `[4,1,2]` | `True` |
| `[3,4,5,1,2,null,null,null,null,0]` | `[4,1,2]` | `False` |

**Constraints:** số node của `root` trong `[1, 2000]`, của `subRoot` trong `[1, 1000]`; giá trị trong `[-10^4, 10^4]`.

## 2. Nền tảng: `TreeNode`, dựng cây, vẽ cây

`build_tree` dựng cây từ list level order (`None` = trống) bằng queue `deque`; `draw` in cây xoay 90° (nhánh phải ở trên).

In [1]:
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left: "Optional[TreeNode]" = None, right: "Optional[TreeNode]" = None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def draw(node: Optional[TreeNode], depth: int = 0) -> None:
    if node is None:
        return
    draw(node.right, depth + 1)
    print("    " * depth + str(node.val))
    draw(node.left, depth + 1)

null = None
root1 = build_tree([3, 4, 5, 1, 2])
root2 = build_tree([3, 4, 5, 1, 2, null, null, null, null, 0])
sub = build_tree([4, 1, 2])
draw(root2)

    5
3
        2
            0
    4
        1


## 3. Viên gạch đầu tiên: hai cây có giống hệt nhau không? (`is_same`)

Gợi ý của LeetCode: hai cây `s`, `t` **giống hệt** nếu
- cả hai rỗng → giống;
- một rỗng, một không → khác;
- giá trị gốc bằng nhau **và** cây con trái giống nhau **và** cây con phải giống nhau.

```text
is_same(s, t) = s.val == t.val AND is_same(s.left, t.left) AND is_same(s.right, t.right)
```

In [2]:
def is_same(a: Optional[TreeNode], b: Optional[TreeNode]) -> bool:
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    return a.val == b.val and is_same(a.left, b.left) and is_same(a.right, b.right)

print(is_same(root1.left, sub))   # True: cây con gốc 4 của root1
print(is_same(root2.left, sub))   # False: có thêm node 0
print(is_same(None, None), is_same(sub, None))

True
False
True False


`and` trong Python **dừng sớm** (short-circuit): nếu `a.val != b.val` thì không gọi đệ quy hai bên nữa — tiết kiệm thời gian.

## 4. Cách 1 — Thử `is_same` tại mọi node của `root` (lời giải chính)

`subRoot` là cây con của `root` khi và chỉ khi **có một node `p` trong `root`** sao cho `is_same(p, subRoot)`.

Duyệt mọi node bằng đệ quy:

```text
is_subtree(root, sub) =
    False                                  nếu root rỗng
    True                                   nếu is_same(root, sub)
    is_subtree(root.left, sub) OR is_subtree(root.right, sub)   ngược lại
```

In [3]:
def is_subtree_dfs(root: Optional[TreeNode], sub: Optional[TreeNode]) -> bool:
    if root is None:
        return sub is None
    if is_same(root, sub):
        return True
    return is_subtree_dfs(root.left, sub) or is_subtree_dfs(root.right, sub)

print(is_subtree_dfs(root1, sub))   # True
print(is_subtree_dfs(root2, sub))   # False

True
False


## 5. Trace Cách 1

Ví dụ 2: `root = [3,4,5,1,2,null,null,null,null,0]`, `sub = [4,1,2]`.

| Node `p` thử | `is_same(p, sub)`? | Lý do |
|---:|:---:|---|
| 3 | ✘ | `3 ≠ 4` |
| 4 | ✘ | gốc, trái `1` khớp; phải: `2` có con trái `0`, còn `sub` thì `None` |
| 1 | ✘ | `1 ≠ 4` |
| 2 | ✘ | `2 ≠ 4` |
| 0 | ✘ | `0 ≠ 4` |
| 5 | ✘ | `5 ≠ 4` |

Không node nào khớp → `False`.

In [4]:
def trace_is_subtree(root, sub):
    def same(a, b, depth):
        pad = "      " + "  " * depth
        if a is None and b is None:
            return True
        if a is None or b is None:
            print(f"{pad}so {getattr(a, 'val', None)} với {getattr(b, 'val', None)}: một bên rỗng -> khác")
            return False
        if a.val != b.val:
            print(f"{pad}so {a.val} với {b.val}: khác giá trị")
            return False
        print(f"{pad}so {a.val} với {b.val}: bằng, xét tiếp con")
        return same(a.left, b.left, depth + 1) and same(a.right, b.right, depth + 1)

    def go(node):
        if node is None:
            return False
        print(f"Thử đỉnh {node.val}:")
        if same(node, sub, 0):
            print(f"  => khớp tại {node.val}")
            return True
        return go(node.left) or go(node.right)

    ans = go(root)
    print("Kết quả:", ans)
    return ans

trace_is_subtree(root2, sub)

Thử đỉnh 3:
      so 3 với 4: khác giá trị
Thử đỉnh 4:
      so 4 với 4: bằng, xét tiếp con
        so 1 với 1: bằng, xét tiếp con
        so 2 với 2: bằng, xét tiếp con
          so 0 với None: một bên rỗng -> khác
Thử đỉnh 1:
      so 1 với 4: khác giá trị
Thử đỉnh 2:
      so 2 với 4: khác giá trị
Thử đỉnh 0:
      so 0 với 4: khác giá trị
Thử đỉnh 5:
      so 5 với 4: khác giá trị
Kết quả: False


False

`getattr(a, 'val', None)` đọc thuộc tính `val`, nếu `a` là `None` (không có `val`) thì trả giá trị mặc định `None` — tiện để in.

In [5]:
trace_is_subtree(root1, sub)

Thử đỉnh 3:
      so 3 với 4: khác giá trị
Thử đỉnh 4:
      so 4 với 4: bằng, xét tiếp con
        so 1 với 1: bằng, xét tiếp con
        so 2 với 2: bằng, xét tiếp con
  => khớp tại 4
Kết quả: True


True

## 6. Code nộp LeetCode — Cách 1

In [6]:
class Solution:
    def isSubtree(self, root: Optional[TreeNode], subRoot: Optional[TreeNode]) -> bool:
        def same(a, b):
            if a is None and b is None:
                return True
            if a is None or b is None or a.val != b.val:
                return False
            return same(a.left, b.left) and same(a.right, b.right)

        def dfs(node):
            if node is None:
                return False
            return same(node, subRoot) or dfs(node.left) or dfs(node.right)

        return dfs(root)

sol = Solution()
print(sol.isSubtree(root1, sub), sol.isSubtree(root2, sub))

True False


### Độ phức tạp Cách 1

Gọi `m` = số node của `root`, `n` = số node của `subRoot`.
- Mỗi node của `root` gọi `is_same` tối đa `O(n)` bước → **`O(m·n)`** xấu nhất (`2000 × 1000 = 2·10^6`, chạy tốt).
- Bộ nhớ: `O(h_root + h_sub)` cho call stack.

Xấu nhất xảy ra khi rất nhiều node "gần khớp" (ví dụ cây toàn giá trị giống nhau). Có thể tốt hơn không? → Mục 7, 9.

## 7. Cách 2 — Tuần tự hóa thành chuỗi rồi tìm chuỗi con

Ý tưởng: biến mỗi cây thành **một chuỗi duy nhất** sao cho

> `subRoot` là cây con của `root` ⇔ chuỗi của `subRoot` là **chuỗi con** của chuỗi của `root`.

Dùng duyệt **tiền thứ tự** (preorder: gốc → trái → phải) và **ghi rõ vị trí `None`** bằng ký hiệu `#`. Trong preorder, một cây con luôn chiếm một **đoạn liên tiếp** của chuỗi.

Hai chi tiết bắt buộc:
1. **Phải ghi `None`** — nếu không, các cây khác hình dạng cho cùng chuỗi.
2. **Mỗi giá trị cần dấu phân cách ở đầu** (ví dụ `,`) — nếu không `"2"` sẽ khớp nhầm bên trong `"12"`.

In [7]:
def serialize(node: Optional[TreeNode]) -> str:
    parts = []
    def pre(n):
        if n is None:
            parts.append(",#")
            return
        parts.append("," + str(n.val))
        pre(n.left)
        pre(n.right)
    pre(node)
    return "".join(parts)

print("root1 :", serialize(root1))
print("root2 :", serialize(root2))
print("sub   :", serialize(sub))
print("sub in root1?", serialize(sub) in serialize(root1))
print("sub in root2?", serialize(sub) in serialize(root2))

root1 : ,3,4,1,#,#,2,#,#,5,#,#
root2 : ,3,4,1,#,#,2,0,#,#,#,5,#,#
sub   : ,4,1,#,#,2,#,#
sub in root1? True
sub in root2? False


Đọc chuỗi `sub`: `,4,1,#,#,2,#,#` — gốc 4; con trái 1 (hai con rỗng); con phải 2 (hai con rỗng). Trong `root2`, đoạn sau `,2` là `,0,...` chứ không phải `,#,#` → không khớp. Đúng!

### Vì sao cần dấu phân cách ở đầu?

In [8]:
def serialize_no_sep(node):
    if node is None:
        return "#"
    return str(node.val) + serialize_no_sep(node.left) + serialize_no_sep(node.right)

big = build_tree([12])
small = build_tree([2])
print(repr(serialize_no_sep(small)), "in", repr(serialize_no_sep(big)), "->",
      serialize_no_sep(small) in serialize_no_sep(big), "(SAI: [2] không phải cây con của [12])")
print(repr(serialize(small)), "in", repr(serialize(big)), "->", serialize(small) in serialize(big))

'2##' in '12##' -> True (SAI: [2] không phải cây con của [12])
',2,#,#' in ',12,#,#' -> False


### Vì sao phải ghi `None`?

In [9]:
def serialize_no_null(node):
    if node is None:
        return ""
    return "," + str(node.val) + serialize_no_null(node.left) + serialize_no_null(node.right)

print(serialize_no_null(root2), "|", serialize_no_null(sub))
print("bỏ None ->", serialize_no_null(sub) in serialize_no_null(root2), "(SAI: đáp án đúng là False)")

,3,4,1,2,0,5 | ,4,1,2
bỏ None -> True (SAI: đáp án đúng là False)


In [10]:
def is_subtree_string(root: Optional[TreeNode], sub: Optional[TreeNode]) -> bool:
    return serialize(sub) in serialize(root)

print(is_subtree_string(root1, sub), is_subtree_string(root2, sub))

True False


### Độ phức tạp Cách 2

- Tuần tự hóa: `O(m + n)` (mỗi node và mỗi vị trí `None` một lần; cây `k` node có `k + 1` vị trí `None`).
- `in` trên chuỗi của CPython dùng thuật toán tìm kiếm tối ưu, thực tế rất nhanh; nếu muốn **đảm bảo** `O(m + n)` xấu nhất thì dùng **KMP** (mục 8).
- Bộ nhớ: `O(m + n)` cho hai chuỗi.

### Code nộp LeetCode — Cách 2

In [11]:
class Solution:
    def isSubtree(self, root: Optional[TreeNode], subRoot: Optional[TreeNode]) -> bool:
        def ser(node, out):
            if node is None:
                out.append(",#")
                return
            out.append("," + str(node.val))
            ser(node.left, out)
            ser(node.right, out)

        a, b = [], []
        ser(root, a)
        ser(subRoot, b)
        return "".join(b) in "".join(a)

print(Solution().isSubtree(root1, sub), Solution().isSubtree(root2, sub))

True False


## 8. (Mở rộng) KMP trên danh sách token — đảm bảo `O(m + n)`

KMP (Knuth–Morris–Pratt) tìm mẫu trong văn bản trong thời gian tuyến tính nhờ bảng **lps** ("longest proper prefix which is also suffix"): khi so khớp thất bại, ta biết có thể "nhảy" mẫu bao xa mà không cần lùi văn bản.

Thay vì chuỗi, ta so trên **list token** (`[4, 1, None, None, 2, None, None]`) — không cần lo dấu phân cách.

In [12]:
def preorder_tokens(node) -> list:
    out = []
    stack = [node]
    while stack:
        n = stack.pop()
        if n is None:
            out.append(None)
            continue
        out.append(n.val)
        stack.append(n.right)     # đẩy phải trước để trái được lấy ra trước
        stack.append(n.left)
    return out

def build_lps(p: list) -> list:
    lps = [0] * len(p)
    k = 0
    for i in range(1, len(p)):
        while k > 0 and p[i] != p[k]:
            k = lps[k - 1]
        if p[i] == p[k]:
            k += 1
        lps[i] = k
    return lps

def kmp_contains(text: list, p: list) -> bool:
    lps = build_lps(p)
    k = 0
    for x in text:
        while k > 0 and x != p[k]:
            k = lps[k - 1]
        if x == p[k]:
            k += 1
            if k == len(p):
                return True
    return False

def is_subtree_kmp(root, sub) -> bool:
    return kmp_contains(preorder_tokens(root), preorder_tokens(sub))

print(preorder_tokens(sub))
print(build_lps([1, 1, 2, 1, 1, 3]))
print(is_subtree_kmp(root1, sub), is_subtree_kmp(root2, sub))

[4, 1, None, None, 2, None, None]
[0, 1, 0, 1, 2, 0]
True False


Ở đây ta duyệt preorder bằng **stack** (đẩy phải trước, trái sau) để tránh đệ quy sâu.

## 9. Cách 3 — Băm cây (Merkle hashing / định danh cây con)

Ý tưởng: gán cho mỗi cây con một **mã định danh** (ID) sao cho hai cây con **giống hệt nhau ⇔ cùng ID**. Khi đó chỉ cần kiểm tra ID của `subRoot` có xuất hiện trong tập ID các cây con của `root` không.

Cách gán ID an toàn (không va chạm): dùng một **dict** chung ánh xạ bộ ba `(val, id_trái, id_phải)` → số nguyên mới. Hai cây con giống hệt nhau có cùng bộ ba (theo quy nạp) nên nhận cùng ID.

`None` có ID `0`.

In [13]:
def is_subtree_hash(root: Optional[TreeNode], sub: Optional[TreeNode]) -> bool:
    table = {}                 # (val, id_left, id_right) -> id

    def get_id(node, collect):
        if node is None:
            return 0
        key = (node.val, get_id(node.left, collect), get_id(node.right, collect))
        if key not in table:
            table[key] = len(table) + 1
        node_id = table[key]
        if collect is not None:
            collect.add(node_id)
        return node_id

    root_ids = set()
    get_id(root, root_ids)
    return get_id(sub, None) in root_ids

print(is_subtree_hash(root1, sub), is_subtree_hash(root2, sub))

True False


In [14]:
def show_ids(root, sub):
    table = {}
    def get_id(node, name):
        if node is None:
            return 0
        key = (node.val, get_id(node.left, name), get_id(node.right, name))
        new = key not in table
        if new:
            table[key] = len(table) + 1
        print(f"[{name}] node {node.val}: key={key} -> id {table[key]}{' (mới)' if new else ' (đã có)'}")
        return table[key]
    get_id(root, "root")
    print("id của sub =", get_id(sub, "sub"))

show_ids(root1, sub)

[root] node 1: key=(1, 0, 0) -> id 1 (mới)
[root] node 2: key=(2, 0, 0) -> id 2 (mới)
[root] node 4: key=(4, 1, 2) -> id 3 (mới)
[root] node 5: key=(5, 0, 0) -> id 4 (mới)
[root] node 3: key=(3, 3, 4) -> id 5 (mới)
[sub] node 1: key=(1, 0, 0) -> id 1 (đã có)
[sub] node 2: key=(2, 0, 0) -> id 2 (đã có)
[sub] node 4: key=(4, 1, 2) -> id 3 (đã có)
id của sub = 3


Ở `root1`, cây con gốc `4` có key `(4, id(1), id(2))`. Khi tính `sub`, key y hệt → **"đã có"** → cùng ID → `True`.

Độ phức tạp: mỗi node tạo một tuple 3 phần tử và tra dict `O(1)` trung bình → **`O(m + n)`** thời gian, `O(m + n)` bộ nhớ.

## 10. Vì sao đúng?

**Cách 1:** theo định nghĩa, `subRoot` là cây con của `root` ⇔ tồn tại node `p` trong `root` với cây con gốc `p` giống hệt `subRoot`. `dfs` thử **mọi** node `p` (đệ quy phủ toàn bộ cây), và `is_same` kiểm tra đúng "giống hệt" (cấu trúc + giá trị, kể cả vị trí `None`) bằng quy nạp ⇒ đúng.

**Cách 2:** với preorder **có ghi `None`**, chuỗi của một cây xác định **duy nhất** cây đó (có thể dựng lại). Cây con gốc `p` chiếm đúng một đoạn liên tiếp trong chuỗi của `root`. Chiều ngược lại: một đoạn bắt đầu tại ranh giới token mà bằng chuỗi hoàn chỉnh của `subRoot` thì chính là chuỗi của cây con tại node đó (vì chuỗi preorder có `None` "tự đóng": số `#` vừa đủ khi cây kết thúc). Dấu `,` ở đầu mỗi token đảm bảo khớp chỉ xảy ra **tại ranh giới token**.

**Cách 3:** quy nạp: hai cây con giống hệt ⇔ cùng `val` và hai con trái giống hệt, hai con phải giống hệt ⇔ cùng bộ ba key ⇔ cùng ID (dict cấp ID duy nhất cho mỗi key).

## 11. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. `is_same` tại mọi node | đệ quy lồng | `O(m·n)` | `O(h)` | **dễ hiểu, chuẩn phỏng vấn** |
| 2. Chuỗi preorder + `in` | string matching | ~`O(m + n)` thực tế | `O(m + n)` | nhớ `#` và dấu phân cách |
| 2b. Token + KMP | string matching đảm bảo | `O(m + n)` | `O(m + n)` | tuyến tính chắc chắn |
| 3. Băm cây (ID) | Merkle hashing | `O(m + n)` trung bình | `O(m + n)` | dùng lại cho bài "duplicate subtrees" |

## 12. Bộ test: mẫu đề, biên, đối chiếu ngẫu nhiên

In [15]:
tests = [
    ([3, 4, 5, 1, 2], [4, 1, 2], True),
    ([3, 4, 5, 1, 2, null, null, null, null, 0], [4, 1, 2], False),
    ([1], [1], True),
    ([1], [2], False),
    ([12], [2], False),                       # bẫy dấu phân cách
    ([1, 1], [1], True),                      # lá 1 là cây con
    ([1, 2, 3], [1, 2], False),               # giống phần đầu nhưng root có thêm con phải
    ([1, 2, 3], [1, 2, 3], True),             # cả cây là cây con của chính nó
    ([1, null, 2, null, 3], [2, null, 3], True),
    ([1, 2, null, 3], [2, null, 3], False),   # khác vị trí trái/phải
    ([3, 4, 5, 1, null, 2], [3, 1, 2], False),
]
methods = [is_subtree_dfs, is_subtree_string, is_subtree_kmp, is_subtree_hash]
for r_vals, s_vals, expected in tests:
    r, s = build_tree(r_vals), build_tree(s_vals)
    for f in methods:
        assert f(r, s) == expected, (f.__name__, r_vals, s_vals)
    print(f"{str(r_vals):<38} {str(s_vals):<14} -> {expected}")
print("Test mẫu PASS!")

[3, 4, 5, 1, 2]                        [4, 1, 2]      -> True
[3, 4, 5, 1, 2, None, None, None, None, 0] [4, 1, 2]      -> False
[1]                                    [1]            -> True
[1]                                    [2]            -> False
[12]                                   [2]            -> False
[1, 1]                                 [1]            -> True
[1, 2, 3]                              [1, 2]         -> False
[1, 2, 3]                              [1, 2, 3]      -> True
[1, None, 2, None, 3]                  [2, None, 3]   -> True
[1, 2, None, 3]                        [2, None, 3]   -> False
[3, 4, 5, 1, None, 2]                  [3, 1, 2]      -> False
Test mẫu PASS!


In [16]:
import random

def random_tree(n: int, vals=(1, 2)) -> Optional[TreeNode]:
    if n == 0:
        return None
    nodes = [TreeNode(random.choice(vals))]
    for _ in range(1, n):
        while True:
            p = random.choice(nodes)
            side = random.choice(("left", "right"))
            if getattr(p, side) is None:
                child = TreeNode(random.choice(vals))
                setattr(p, side, child)
                nodes.append(child)
                break
    return nodes[0]

def all_nodes(root):
    out, stack = [], [root]
    while stack:
        n = stack.pop()
        if n:
            out.append(n)
            stack.extend((n.left, n.right))
    return out

random.seed(572)
true_count = 0
for _ in range(500):
    r = random_tree(random.randint(1, 25))
    # một nửa số lần lấy đúng một cây con của r, nửa còn lại sinh ngẫu nhiên
    s = random.choice(all_nodes(r)) if random.random() < 0.5 else random_tree(random.randint(1, 5))
    answers = {f(r, s) for f in methods}
    assert len(answers) == 1, answers
    true_count += answers.pop()
print(f"500 cặp cây ngẫu nhiên (giá trị chỉ 1 và 2 để dễ 'gần khớp'): 4 cách khớp nhau; {true_count} cặp True.")

500 cặp cây ngẫu nhiên (giá trị chỉ 1 và 2 để dễ 'gần khớp'): 4 cách khớp nhau; 310 cặp True.


Chọn giá trị chỉ trong `{1, 2}` để tạo nhiều trường hợp "gần giống" — đây là nơi các lời giải sai dễ lộ.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ so sánh gốc hoặc chỉ so "phần đầu" (bỏ qua con thừa)

In [17]:
def wrong_prefix_match(a, b):
    # SAI: coi như khớp khi b hết node, bất kể a còn con
    if b is None:
        return True
    if a is None or a.val != b.val:
        return False
    return wrong_prefix_match(a.left, b.left) and wrong_prefix_match(a.right, b.right)

def wrong_is_subtree(root, sub):
    if root is None:
        return False
    return wrong_prefix_match(root, sub) or wrong_is_subtree(root.left, sub) or wrong_is_subtree(root.right, sub)

print("sai:", wrong_is_subtree(root2, sub), "| đúng:", is_subtree_dfs(root2, sub))

sai: True | đúng: False


Vì sao sai: đó là kiểm tra "`sub` là **phần trên** của một nhánh", không phải cây con. Định nghĩa cây con yêu cầu lấy **toàn bộ hậu duệ**. Sửa: `is_same` phải trả `False` khi **một bên** rỗng.

### Lỗi 2: Gọi `is_subtree` đệ quy thay vì `is_same` khi so hai con

Viết `root.val == sub.val and is_subtree(root.left, sub.left) and ...` — trộn lẫn hai hàm → cho phép con khớp ở **sâu hơn**, sai cấu trúc.

In [18]:
def wrong_mixed(root, sub):
    if sub is None:
        return True
    if root is None:
        return False
    if root.val == sub.val and wrong_mixed(root.left, sub.left) and wrong_mixed(root.right, sub.right):
        return True
    return wrong_mixed(root.left, sub) or wrong_mixed(root.right, sub)

r = build_tree([1, 2, null, 3])
s = build_tree([1, 3])
print("sai:", wrong_mixed(r, s), "| đúng:", is_subtree_dfs(r, s))

sai: True | đúng: False


Ở ví dụ trên, `3` nằm ở cháu chứ không phải con trái trực tiếp của `1`, nhưng hàm trộn lại "nhảy" xuống sâu hơn để tìm.

### Lỗi 3: Tuần tự hóa thiếu `None` hoặc thiếu dấu phân cách

Đã minh họa ở mục 7: `[2]` "khớp" `[12]`, và ví dụ 2 của đề cho `True` sai.

### Lỗi 4: Tuần tự hóa bằng inorder

Inorder (trái → gốc → phải) **không** đảm bảo cây con là đoạn liên tiếp "tự đóng" như preorder có `None` — dễ khớp nhầm. Dùng preorder (hoặc postorder) có ghi `None`.

### Lỗi 5: Dùng `hash()` tự chế dạng tổng/nhân rồi so bằng hash mà không xác nhận

Hash tự chế (ví dụ cộng giá trị) có thể **va chạm**: hai cây khác nhau cùng hash. Hoặc dùng dict cấp ID như Cách 3 (không va chạm), hoặc so hash xong phải kiểm tra lại bằng `is_same`.

In [19]:
def weak_hash(node):
    if node is None:
        return 0
    return node.val + weak_hash(node.left) + weak_hash(node.right)

t1 = build_tree([3, 1, 2])
t2 = build_tree([3, 2, 1])
print("weak_hash bằng nhau:", weak_hash(t1) == weak_hash(t2), "| nhưng is_same:", is_same(t1, t2))

weak_hash bằng nhau: True | nhưng is_same: False


## 14. Mẹo nhận dạng pattern và biến thể

### Mẹo
- "Cây A có chứa cây B không / hai cây có giống nhau không" → viết `is_same` trước, rồi duyệt.
- So sánh **cấu trúc** cây nhanh → **tuần tự hóa có `None`** hoặc **cấp ID cho cây con**.
- Tìm mẫu trong chuỗi/list trong thời gian tuyến tính → **KMP**.

### Biến thể
1. **LeetCode 100 – Same Tree**: chính là `is_same`.
2. **LeetCode 101 – Symmetric Tree**: so cây với ảnh gương của nó.
3. **LeetCode 652 – Find Duplicate Subtrees** (Medium): cấp ID/tuần tự hóa mọi cây con, đếm trùng.
4. **LeetCode 1367 – Linked List in Binary Tree** (Medium): khớp "đường đi xuống" (ở đó khớp phần đầu lại là **đúng**!).
5. **LeetCode 297 – Serialize and Deserialize Binary Tree** (Hard).

In [20]:
# Biến thể 3 demo: tìm các cây con lặp lại bằng cấp ID
def find_duplicate_subtrees(root):
    table, count, result = {}, {}, []
    def get_id(node):
        if node is None:
            return 0
        key = (node.val, get_id(node.left), get_id(node.right))
        if key not in table:
            table[key] = len(table) + 1
        i = table[key]
        count[i] = count.get(i, 0) + 1
        if count[i] == 2:
            result.append(serialize(node))
        return i
    get_id(root)
    return result

print(find_duplicate_subtrees(build_tree([1, 2, 3, 4, null, 2, 4, null, null, 4])))

[',4,#,#', ',2,4,#,#,#']


Kết quả là chuỗi tuần tự hóa của hai cây con lặp lại: `[4]` và `[2,4]`.

## 15. Bài tập tự luyện

### Bài 1 (dễ)
Viết `count_matches(root, sub)`: đếm số node `p` trong `root` mà cây con gốc `p` giống hệt `sub`.

In [21]:
def count_matches(root, sub) -> int:
    # TODO
    pass

# assert count_matches(build_tree([1, 1, 1]), build_tree([1])) == 2
# assert count_matches(root1, sub) == 1

### Bài 2 (dễ)
Viết `is_mirror_subtree(root, sub)`: có cây con nào của `root` là **ảnh gương** của `sub` không?

In [22]:
def is_mirror_subtree(root, sub) -> bool:
    # TODO
    pass

# assert is_mirror_subtree(build_tree([3, 4, 5, 2, 1]), build_tree([4, 1, 2])) == True
# assert is_mirror_subtree(root1, sub) == False

### Bài 3 (vừa)
Viết `deserialize(s)`: dựng lại cây từ chuỗi do `serialize` sinh ra (dạng `",4,1,#,#,2,#,#"`).

In [23]:
def deserialize(s: str):
    # TODO: tách token, dùng iterator + đệ quy preorder
    pass

# assert serialize(deserialize(serialize(root2))) == serialize(root2)

### Bài 4 (vừa)
Viết `is_subtree_bfs(root, sub)`: như Cách 1 nhưng duyệt `root` bằng **BFS (queue)** và chỉ gọi `is_same` khi `node.val == sub.val`.

In [24]:
def is_subtree_bfs(root, sub) -> bool:
    # TODO
    pass

# assert is_subtree_bfs(root1, sub) == True
# assert is_subtree_bfs(root2, sub) == False

### Bài 5 (khó hơn)
Viết `largest_common_subtree_size(a, b)`: số node của cây con **lớn nhất** xuất hiện (giống hệt) trong cả hai cây `a` và `b` (0 nếu không có).

In [25]:
def largest_common_subtree_size(a, b) -> int:
    # TODO: cấp ID chung cho cả hai cây, kèm kích thước
    pass

# assert largest_common_subtree_size(root1, root2) == 1
# assert largest_common_subtree_size(root1, build_tree([9, 4, 7, 1, 2])) == 3

## 16. Gợi ý 3 mức

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Giống `dfs` | Không dừng khi khớp | `int(is_same(...)) + count(left) + count(right)` |
| 2 | Viết `is_mirror(a, b)` | So `a.left` với `b.right` | Duyệt mọi node như Cách 1 |
| 3 | `s.split(",")[1:]` | `iter(tokens)` + `next` | `#` → `None`; số → node, dựng trái rồi phải |
| 4 | `deque([root])` | `popleft`, kiểm tra giá trị | Thêm con vào queue |
| 5 | Dict key `(val, idL, idR)` dùng chung | Lưu `size[id]` | Giao hai tập ID, lấy `max(size)` |

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [26]:
def answer_count_matches(root, sub) -> int:
    if root is None:
        return 0
    return int(is_same(root, sub)) + answer_count_matches(root.left, sub) + answer_count_matches(root.right, sub)

def is_mirror(a, b) -> bool:
    if a is None and b is None:
        return True
    if a is None or b is None or a.val != b.val:
        return False
    return is_mirror(a.left, b.right) and is_mirror(a.right, b.left)

def answer_is_mirror_subtree(root, sub) -> bool:
    if root is None:
        return False
    return is_mirror(root, sub) or answer_is_mirror_subtree(root.left, sub) or answer_is_mirror_subtree(root.right, sub)

def answer_deserialize(s: str):
    tokens = iter(s.split(",")[1:])
    def build():
        t = next(tokens)
        if t == "#":
            return None
        node = TreeNode(int(t))
        node.left = build()
        node.right = build()
        return node
    return build()

def answer_is_subtree_bfs(root, sub) -> bool:
    q = deque([root])
    while q:
        node = q.popleft()
        if node is None:
            continue
        if node.val == sub.val and is_same(node, sub):
            return True
        q.append(node.left)
        q.append(node.right)
    return False

def answer_largest_common_subtree_size(a, b) -> int:
    table, size = {}, {0: 0}
    def get_id(node, bucket):
        if node is None:
            return 0
        l, r = get_id(node.left, bucket), get_id(node.right, bucket)
        key = (node.val, l, r)
        if key not in table:
            table[key] = len(table) + 1
            size[table[key]] = 1 + size[l] + size[r]
        bucket.add(table[key])
        return table[key]
    ids_a, ids_b = set(), set()
    get_id(a, ids_a)
    get_id(b, ids_b)
    common = ids_a & ids_b
    return max((size[i] for i in common), default=0)

assert answer_count_matches(build_tree([1, 1, 1]), build_tree([1])) == 2
assert answer_count_matches(root1, sub) == 1
assert answer_is_mirror_subtree(build_tree([3, 4, 5, 2, 1]), build_tree([4, 1, 2])) == True
assert answer_is_mirror_subtree(root1, sub) == False
assert serialize(answer_deserialize(serialize(root2))) == serialize(root2)
assert answer_is_subtree_bfs(root1, sub) == True
assert answer_is_subtree_bfs(root2, sub) == False
assert answer_largest_common_subtree_size(root1, root2) == 1                     # chỉ lá 1 (hoặc 5) chung
assert answer_largest_common_subtree_size(root1, build_tree([9, 4, 7, 1, 2])) == 3
for _ in range(200):
    r = random_tree(random.randint(1, 20))
    s = random_tree(random.randint(1, 4))
    assert answer_is_subtree_bfs(r, s) == is_subtree_dfs(r, s)
    assert serialize(answer_deserialize(serialize(r))) == serialize(r)
print("Đáp án chạy PASS toàn bộ test!")

Đáp án chạy PASS toàn bộ test!


`ids_a & ids_b` là **giao** của hai set. `max(..., default=0)` an toàn khi giao rỗng.

## 18. Template nhớ nhanh và checklist

```python
class Solution:
    def isSubtree(self, root, subRoot) -> bool:
        def same(a, b):
            if not a and not b:
                return True
            if not a or not b or a.val != b.val:
                return False
            return same(a.left, b.left) and same(a.right, b.right)

        def dfs(node):
            return node is not None and (same(node, subRoot) or dfs(node.left) or dfs(node.right))

        return dfs(root)
```

Bản tuyến tính: `serialize(sub) in serialize(root)` với preorder có `#` và dấu `,` **trước** mỗi token.

### Checklist trước khi submit
- [ ] `same` trả `False` khi **chỉ một** bên rỗng (không cho phép "khớp phần đầu").
- [ ] `same` so con trái–trái, phải–phải bằng **`same`**, không gọi lại `isSubtree`.
- [ ] Thử mọi node của `root`, kể cả chính `root` và các lá.
- [ ] Nếu tuần tự hóa: có ký hiệu `None` và dấu phân cách đầu token.
- [ ] Đã thử ví dụ 2 của đề (cây con có thêm node con).